bar_impact.ipynb
───────────────────────────
Overlay of u(t), v(t), and impulses from three integration schemes (NS-Newmark, CD-Lagrange, Moreau-Jean) on a single set of plots with identical parameters. Includes analytical u(t), v(t).

In [ ]:
from __future__ import annotations
from dataclasses import dataclass
from typing import Dict, Tuple
import numpy as np
import scipy.sparse as sp
import scipy.sparse.linalg as spla
import plotly.graph_objects as go
import plotly.io as pio

pio.renderers.default = "notebook_connected"

In [ ]:
@dataclass
class BarParams:
    rho: float = 7800.0  # Density [kg/m^3] (Steel)
    S: float = 0.01  # Cross-section Area [m^2]
    E: float = 210e9  # Young's Modulus [Pa]
    L: float = 1.0  # Length [m]
    u0: float = 0.0  # Initial displacement [m]
    v0: float = -1.0  # Initial velocity [m/s] (towards wall)
    e: float = 0.5  # Restitution coefficient in [0,1]
    N: int = 50  # Number of elements
    t_end: float = 0.005  # End time [s]
    dt_factor: float = 0.9  # Factor to scale stable dt

    @property
    def l_elem(self) -> float:
        return self.L / self.N

    @property
    def m_elem(self) -> float:
        return self.rho * self.S * self.l_elem

    @property
    def k_elem(self) -> float:
        return self.E * self.S / self.l_elem

    @property
    def c_wave(self) -> float:
        return np.sqrt(self.E / self.rho)

    @property
    def dt_stable(self) -> float:
        # CFL condition for explicit schemes: dt <= l/c
        return self.dt_factor * self.l_elem / self.c_wave


def build_matrices(p: BarParams) -> Tuple[sp.csc_matrix, sp.csc_matrix]:
    """
    Constructs lumped Mass (M) and Stiffness (K) matrices for 1D linear elements.
    Nodes: 0 to N. Node 0 is the contact node (left). Node N is the free end (right).
    """
    nnz = p.N + 1

    # Lumped Mass Matrix (Diagonal)
    # Interior nodes get m_elem, boundary nodes get m_elem/2
    m_diag = np.full(nnz, p.m_elem)
    m_diag[0] = p.m_elem / 2.0
    m_diag[-1] = p.m_elem / 2.0
    M = sp.diags(m_diag, format="csc")

    # Stiffness Matrix (Tridiagonal)
    # [ 1 -1  0 ]
    # [-1  2 -1 ] * k_elem
    # [ 0 -1  1 ]
    diagonals = [np.full(nnz, 2.0), np.full(nnz - 1, -1.0), np.full(nnz - 1, -1.0)]
    diagonals[0][0] = 1.0
    diagonals[0][-1] = 1.0
    K = sp.diags(diagonals, [0, -1, 1], format="csc") * p.k_elem

    return M, K

In [ ]:
# ─── Plot layout helper ───


def get_layout() -> go.Layout:

    layout = go.Layout(
        xaxis=dict(
            showgrid=True,
            showline=True,
            linewidth=1,
            linecolor="black",
            mirror=True,
            zeroline=False,
            ticks="inside",
            exponentformat="power",
            showexponent="last",
            tickfont=dict(size=12),
        ),
        yaxis=dict(
            showgrid=True,
            showline=True,
            linewidth=1,
            linecolor="black",
            mirror=True,
            zeroline=False,
            ticks="inside",
            exponentformat="power",
            tickfont=dict(size=12),
        ),
        font=dict(family="Latin-Modern", size=12, color="Black"),
        legend=dict(
            x=0.97,  # position from the left (0 to 1)
            y=1.2,  # position from the bottom (0 to 1)
            bgcolor="rgba(255, 255, 255, 0.8)",  # white with 80% opacity
            bordercolor="black",
            borderwidth=1,
            orientation="h",
            xanchor="right",
            yanchor="top",
        ),
        width=600,
        height=500,
        showlegend=True,
        template="plotly_white",
    )

    return layout

In [ ]:
def simulate_ns_newmark_bar(p: BarParams) -> Dict[str, np.ndarray]:
    """
    Simplified General NSN Scheme (Scalar Implementation for Single Contact).
    """
    M, K = build_matrices(p)  # noqa: N806
    M_inv_diag = 1.0 / M.diagonal()  # noqa: N806
    M_inv = sp.diags(M_inv_diag, format="csc")  # noqa: N806
    dt = p.dt_stable
    n_steps = int(np.ceil(p.t_end / dt))

    # ─── Pre-computation (Scalar Reduction) ───
    m_inv_0 = M_inv_diag[0]
    k_val_0 = K[0, 0]

    # Modified Delassus Operator W'
    W_prime = m_inv_0 - (dt**2 / 4.0) * m_inv_0 * k_val_0 * m_inv_0  # noqa: N806

    # State Initialization
    u = np.full(p.N + 1, p.u0)
    v = np.full(p.N + 1, p.v0)
    a = -M_inv @ (K @ u)

    T = np.linspace(0, n_steps * dt, n_steps + 1)  # noqa: N806
    U_hist, V_hist, P_hist = (  # noqa: N806
        np.zeros(n_steps + 1),
        np.zeros(n_steps + 1),
        np.zeros(n_steps + 1),
    )
    U_hist[0], V_hist[0], P_hist[0] = u[0], v[0], 0.0

    J_col_flat = (0.5 * dt * m_inv_0) * K[:, 0].toarray().ravel()  # noqa: N806

    for i in range(n_steps):

        # Prediction
        # Displacement Prediction
        u_tilde = u + dt * v + 0.5 * dt**2 * a
        r = K @ u_tilde
        impulse = 0.0

        # Contact Logic
        gap_pred = u_tilde[0]

        # Initialize variables for the update
        w = np.zeros_like(v)  # Velocity jump
        f_contact = np.zeros_like(r)  # Contact force correction (K * correction)

        if gap_pred <= 0.0:
            # Build RHS 'b'
            # q = H( v + 0.5*dt*a + e*v )
            q = v[0] * (1 + p.e) + (0.5 * dt * a[0])

            # b = q - 0.5*dt * H * M^-1 * r
            # Note: r = K*u (internal force).
            b = q - (0.5 * dt * m_inv_0 * r[0])

            # Solve QP for impulse
            # p = max(0, -b / W')
            impulse = max(-b / W_prime, 0.0)

            # Compute corrections
            if impulse > 0:
                # Velocity jump w = M^-1 * H^T * p
                w[0] = impulse * m_inv_0
                f_contact = J_col_flat * impulse

        # Updates
        # Acceleration Update
        # a_next = M^-1 * ( -K*u_next )
        # K*u_next = K*u_tilde + f_contact = r + f_contact
        total_internal_force = r + f_contact
        a_next = -M_inv @ total_internal_force

        # Velocity Update
        # v_next = v_n + (dt/2)*(a_n + a_next) + w
        v_next = v + (0.5 * dt) * (a + a_next) + w

        # Position Update
        u_next = u_tilde + (0.5 * dt) * w

        u, v, a = u_next, v_next, a_next
        U_hist[i + 1], V_hist[i + 1], P_hist[i + 1] = (
            u[0],
            v[0],
            impulse / dt,
        )  # Convert impulse to force

    return {"t": T, "u": U_hist, "v": V_hist, "p": P_hist}


def simulate_cd_lagrange_bar(p: BarParams) -> Dict[str, np.ndarray]:
    """
    Central Difference with Lagrange Multiplier (Impact).
    """
    M, K = build_matrices(p)  # noqa: N806
    M_inv = sp.diags(1.0 / M.diagonal(), format="csc")  # noqa: N806

    dt = p.dt_stable
    n_steps = int(np.ceil(p.t_end / dt))

    u = np.ones(p.N + 1) * p.u0  # Start at gap = 0 (touching)
    v = np.full(p.N + 1, p.v0)

    # CD Start-up: v_half = v_0 + 0.5 * dt * a_0
    a_0 = -M_inv @ (K @ u)
    v_half = v + 0.5 * dt * a_0

    T = np.linspace(0, n_steps * dt, n_steps + 1)  # noqa: N806
    U_hist, V_hist, P_hist = (  # noqa: N806
        np.zeros(n_steps + 1),
        np.zeros(n_steps + 1),
        np.zeros(n_steps + 1),
    )
    U_hist[0], V_hist[0] = u[0], v[0]

    for i in range(n_steps):
        # Update Position
        u_next = u + dt * v_half
        v_half_free = -dt * M_inv @ (K @ u_next) + v_half

        # Check contact at node 0
        if (u_next[0] <= 0.0) and ((v_half_free[0] + p.e * v_half[0]) < 0.0):
            # Impulse required.
            v_half_next = v_half_free.copy()
            v_half_next[0] = -p.e * v_half[0]
            delta_v = v_half_next[0] - v_half_free[0]
            impulse = delta_v * M.diagonal()[0]
        else:
            v_half_next = v_half_free
            impulse = 0.0

        # Centered velocity for output
        v_centered = 0.5 * (v_half + v_half_next)
        u, v_half = u_next, v_half_next

        U_hist[i + 1] = u[0]
        V_hist[i + 1] = v_centered[0]
        P_hist[i + 1] = impulse / dt  # Convert impulse to force

    return {"t": T, "u": U_hist, "v": V_hist, "p": P_hist}


def simulate_moreau_jean_bar(p: BarParams) -> Dict[str, np.ndarray]:
    """
    Moreau-Jean Scheme (Theta-Method / Implicit).
    Uses Theta=0.5 (Trapezoidal) for smooth motion to compare with explicit schemes fairly.
    """
    # ─── Initialization & Constants ───
    M, K = build_matrices(p)  # noqa: N806
    dt = p.dt_stable
    n_steps = int(np.ceil(p.t_end / dt))
    theta = 0.5

    # The implicit LHS matrix coefficient for K: theta * (theta * dt) * dt
    # The explicit RHS matrix coefficient for K acting on v_n: theta * (1 - theta) * dt^2
    coeff_lhs = (theta * dt) ** 2
    coeff_rhs = theta * (1 - theta) * dt**2

    # System Matrices Pre-calculation
    # Left-Hand Side Operator (A): (M + theta^2 * dt^2 * K)
    A = M + coeff_lhs * K  # noqa: N806
    solver = spla.factorized(A)

    # Right-Hand Side Operators (B):
    # Instead of calculating M*v - dt*K*u - coeff*K*v every step,
    # we pre-calculate the linear operators acting on v_n and u_n.
    # RHS = (M - coeff_rhs * K) * v_n - (dt * K) * u_n
    B_v = M - coeff_rhs * K  # noqa: N806
    B_u = -dt * K  # noqa: N806

    # Contact Solver Prep
    # We only monitor contact at the first node (index 0)
    contact_idx = 0

    # Calculate effective inverse mass at the contact node (scalar w)
    e_contact = np.zeros(p.N + 1)
    e_contact[contact_idx] = 1.0

    A_inv_col = solver(
        e_contact
    )  # The column of A^-1 corresponding to contact node #noqa: N806
    w_eff = A_inv_col[contact_idx]  # The diagonal term (scalar)

    # State Initialization
    u = np.full(p.N + 1, p.u0)
    v = np.full(p.N + 1, p.v0)

    # Storage
    T = np.linspace(0, n_steps * dt, n_steps + 1)  # noqa: N806
    U_hist = np.zeros(n_steps + 1)  # noqa: N806
    V_hist = np.zeros(n_steps + 1)  # noqa: N806
    P_hist = np.zeros(n_steps + 1)  # noqa: N806
    U_hist[0], V_hist[0], P_hist[0] = u[contact_idx], v[contact_idx], 0.0

    # Time Stepping Loop
    for i in range(n_steps):

        # Prediction Phase (Unconstrained)
        # Calculate free velocity neglecting contact forces
        rhs_vector = (B_v @ v) + (B_u @ u)
        v_free = solver(rhs_vector)

        # Explicit  prediction for gap detection
        # u_pred = u_n + dt * v_n
        gap_pred = u[contact_idx] + dt * v[contact_idx]

        # Contact Resolution Phase
        v_next = v_free.copy()

        # If the predictor indicates penetration (gap <= 0)
        if gap_pred <= 0.0:
            # Newton's Impact Law: v_new >= -e * v_old
            v_target = -p.e * v[contact_idx]
            v_current = v_free[contact_idx]

            # If the free velocity violates the restitution law (penetrating too fast)
            if v_current < v_target:
                # Calculate required impulse magnitude lambda
                # v_corrected = v_free + lambda * A_inv_col
                # lambda = (v_target - v_free) / w_eff
                impulse_mag = (v_target - v_current) / w_eff

                # Apply correction
                v_next += impulse_mag * A_inv_col
                impulse = impulse_mag
            else:
                impulse = 0.0
        else:
            impulse = 0.0

        # Update Phase
        # Trapezoidal Position Update (Eq 19d)
        v_avg = (1 - theta) * v + theta * v_next
        u_next = u + dt * v_avg

        # Advance state
        u, v = u_next, v_next

        # Store history
        U_hist[i + 1] = u[contact_idx]
        V_hist[i + 1] = v[contact_idx]
        P_hist[i + 1] = impulse / dt  # Convert impulse to force
    return {"t": T, "u": U_hist, "v": V_hist, "p": P_hist}


def analytic_bar_solution(t: np.ndarray, p: BarParams) -> Dict[str, np.ndarray]:
    """
    Exact solution for 1D Bar Impact based on wave propagation.
    """
    # 1. Calculate Wave Speed c0
    # Assuming p has E (Young's Modulus) and rho (Density)
    c0 = np.sqrt(p.E / p.rho)
    Z = p.S * np.sqrt(p.rho * p.E)

    # 2. Key Timings
    # Time to impact: distance / speed
    # (Assuming v0 < 0 and u0 > 0)
    if p.v0 >= 0:
        t_impact = np.inf
    else:
        t_impact = -p.u0 / p.v0

    # Duration of contact: Round-trip wave time (2L / c)
    t_wave_travel = 2 * p.L / c0
    t_separation = t_impact + t_wave_travel

    # 3. Vectorized Construction
    u = np.zeros_like(t)
    v = np.zeros_like(t)
    imp = np.zeros_like(t)

    # -- Phase 1: Free Flight Approach --
    mask_app = t < t_impact
    u[mask_app] = p.u0 + p.v0 * t[mask_app]
    v[mask_app] = p.v0

    # -- Phase 2: Contact (Wave transit) --
    # Node 0 is clamped by the wall
    mask_con = (t >= t_impact) & (t < t_separation)
    u[mask_con] = 0.0
    v[mask_con] = 0.0
    imp[mask_con] = Z * abs(p.v0)  # * p.dt_stable

    # -- Phase 3: Separation (Rebound) --
    mask_sep = t >= t_separation

    # Target rebound velocity
    # In ideal wave theory e=1, but we scale by p.e to compare with discrete schemes
    v_rebound = -p.v0  # -p.e * p.v0

    u[mask_sep] = v_rebound * (t[mask_sep] - t_separation)
    v[mask_sep] = v_rebound

    return {"t": t, "u": u, "v": v, "p": imp}

In [ ]:
# ─── Setup Parameters ───

# Using a finer mesh for the bar to capture wave effects cleanly
N = 50
rho = 7847
L = 0.254
v0 = -5
u0 = 1e-3 * L
E = 2.1e11
dt_factor = 0.7
e = 0.0
S = 0.000645
c = np.sqrt(E / rho)

t_impact = -u0 / v0
time_rebound = 2 * L / np.sqrt(E / rho)

t_end = 2 * u0 / abs(v0) + 6 * L / c
params = BarParams(
    N=N, t_end=t_end, u0=u0, v0=v0, e=e, dt_factor=dt_factor, rho=rho, L=L, E=E, S=S
)

print(f"Running simulations with dt = {params.dt_stable:.3e} s")

# Run Simulations
res_bar = {
    "Moreau-Jean": simulate_moreau_jean_bar(params),
    "CD-Lagrange": simulate_cd_lagrange_bar(params),
    "NS-Newmark": simulate_ns_newmark_bar(params),
}

# Plotting
colors = {"Moreau-Jean": "lightblue", "CD-Lagrange": "#2072b2", "NS-Newmark": "red"}

t_common = res_bar["Moreau-Jean"]["t"]
ana = analytic_bar_solution(t_common, params)

output_name = f"impacting_bar_N{params.N}_e{params.e}_dtf{params.dt_factor}"

In [ ]:
write_output = False

# Contact Node Displacement
fig_u = go.Figure()

fig_u.add_trace(
    go.Scatter(
        x=(ana["t"] - t_impact) / time_rebound,
        y=ana["u"] / L,
        mode="lines",
        name="Analytic (Wave Eq)",
        line=dict(color="black", dash="solid", width=2),
    )
)

for name, data in res_bar.items():
    fig_u.add_trace(
        go.Scatter(
            x=(data["t"] - t_impact) / time_rebound,
            y=data["u"] / L,
            mode="lines",
            name=name,
            line=dict(color=colors[name], width=2),
        )
    )

fig_u.update_layout(get_layout())
fig_u.update_layout(
    title="Impacting Bar: Contact Node Displacement u(0, t)",
    xaxis_title=r"$t / t_b$",
    yaxis_title=r"$u_c / L$",
    template="plotly_white",
    width=700,
    height=500,
    yaxis=dict(showexponent="last", exponentformat="power"),
)
fig_u.show()

if write_output:
    fig_u.write_image(f"../output/{output_name}_u_overlay.pdf", width=400, height=300)

fig_u_zoom = go.Figure()
fig_u_zoom.add_trace(
    go.Scatter(
        x=(ana["t"] - t_impact) / time_rebound,
        y=ana["u"] / L,
        mode="lines",
        name="Analytic (Wave Eq)",
        line=dict(color="black", dash="solid", width=2),
    )
)
for name, data in res_bar.items():
    fig_u_zoom.add_trace(
        go.Scatter(
            x=(data["t"] - t_impact) / time_rebound,
            y=data["u"] / L,
            mode="lines",
            name=name,
            line=dict(color=colors[name], width=2),
        )
    )
fig_u_zoom.update_layout(get_layout())
fig_u_zoom.update_layout(
    title="Impacting Bar: Contact Node Displacement u(0, t) (Zoomed)",
    xaxis_title=r"$t / t_b$",
    yaxis_title=r"$u_c / L$",
    template="plotly_white",
    width=700,
    height=500,
    yaxis=dict(showexponent="last", exponentformat="power"),
    xaxis_range=[0.95, 1.1],
    yaxis_range=[-5e-5, 2e-4],
    showlegend=False,
)
fig_u_zoom.show()
if write_output:
    fig_u_zoom.write_image(
        f"../output/{output_name}_u_overlay_zoom.pdf", width=250, height=250
    )

# Contact Node Velocity
fig_v = go.Figure()

fig_v.add_trace(
    go.Scatter(
        x=(ana["t"] - t_impact) / time_rebound,
        y=ana["v"] * (-1) / v0,
        mode="lines",
        name="Analytic (Wave Eq)",
        line=dict(color="black", dash="solid", width=2),
    )
)

for name, data in res_bar.items():
    fig_v.add_trace(
        go.Scatter(
            x=(data["t"] - t_impact) / time_rebound,
            y=data["v"] * (-1) / v0,
            mode="lines",
            name=name,
            line=dict(color=colors[name], width=2),
        )
    )

fig_v.update_layout(get_layout())
fig_v.update_layout(
    title="Impacting Bar: Contact Node Velocity v(0, t)",
    xaxis_title=r"$t / t_b$",
    yaxis_title=r"$v_c / v_0$",
    template="plotly_white",
    width=700,
    height=500,
)
fig_v.show()

if write_output:
    fig_v.write_image(f"../output/{output_name}_v_overlay.pdf", width=400, height=300)

fig_p = go.Figure()
fig_p.add_trace(
    go.Scatter(
        x=(ana["t"] - t_impact) / time_rebound,
        y=ana["p"] * (-1) / (rho * c * v0 * S),
        mode="lines",
        name="Analytic (Wave Eq)",
        line=dict(color="black", dash="solid", width=2),
    )
)

for name, data in res_bar.items():
    fig_p.add_trace(
        go.Scatter(
            x=(data["t"] - t_impact) / time_rebound,
            y=data["p"] * (-1) / (rho * c * v0 * S),
            mode="lines",
            name=name,
            line=dict(color=colors[name], width=2),
        )
    )
fig_p.update_layout(get_layout())
fig_p.update_layout(
    title="Impacting Bar: Contact Force p(0, t)",
    xaxis_title=r"$t / t_b$",
    yaxis_title=r"$F_c / (\rho c v_0 S)$",
    template="plotly_white",
    width=700,
    height=500,
    yaxis=dict(showexponent="last", exponentformat="power"),
    showlegend=True,
)
fig_p.show()

if write_output:
    fig_p.write_image(f"../output/{output_name}_p_overlay.pdf", width=400, height=300)

fig_p_zoom = go.Figure()
fig_p_zoom.add_trace(
    go.Scatter(
        x=(ana["t"] - t_impact) / time_rebound,
        y=ana["p"] * (-1) / (rho * c * v0 * S),
        mode="lines",
        name="Analytic (Wave Eq)",
        line=dict(color="black", dash="solid", width=2),
    )
)

for name, data in res_bar.items():
    fig_p_zoom.add_trace(
        go.Scatter(
            x=(data["t"] - t_impact) / time_rebound,
            y=data["p"] * (-1) / (rho * c * v0 * S),
            mode="lines",
            name=name,
            line=dict(color=colors[name], width=2),
        )
    )
fig_p_zoom.update_layout(get_layout())
fig_p_zoom.update_layout(
    title="Impacting Bar: Contact Force p(0, t) (Zoomed)",
    xaxis_title=r"$t / t_b$",
    yaxis_title=r"$F_c / (\rho c v_0 S)$",
    template="plotly_white",
    width=700,
    height=500,
    yaxis=dict(showexponent="last"),
    xaxis_range=[-0.02, 0.08],
    yaxis_range=[0.65, 1.35],
    showlegend=False,
)
fig_p_zoom.show()

if write_output:
    fig_p_zoom.write_image(
        f"../output/{output_name}_p_overlay_zoom.pdf", width=250, height=250
    )

### Timestep convergence (after release)

In [ ]:
# ─── Simulation Setup ───

# Physical Constants
rho = 7847.0  # Density [kg/m^3]
E = 2.1e11  # Young's Modulus [Pa]
L = 0.254  # Length [m]
S = 0.000645  # Area [m^2]
c = np.sqrt(E / rho)  # Wave speed

# Impact Conditions
v0 = -5.0  # Initial velocity [m/s]
u0 = 1e-3 * L  # Initial gap [m]
e = 0.0  # Restitution

# Analytical Timing
t_impact = -u0 / v0
t_contact = 2 * L / c
t_release = t_impact + t_contact

# Simulation Horizon (allow some time after release)
t_end = t_release + (6 * L / c)

# ─── Space-Time Sweep Parameters ───
# Range: Coarse (N=10) to Fine (N=1000)
N_vals = np.unique(np.geomspace(1, 5000, 50).astype(int))

methods = {
    "Moreau-Jean": simulate_moreau_jean_bar,
    "CD-Lagrange": simulate_cd_lagrange_bar,
    "NS-Newmark": simulate_ns_newmark_bar,
}

# Storage for results (Now includes err_v)
results = {
    name: {"dt": [], "err_u": [], "err_v": [], "err_u_l2": [], "err_v_l2": []}
    for name in methods
}

print("Running Space-Time Convergence Study (h = c*dt)...")
print(f"Measuring error only for t > {1.1*t_release*1e3:.3f} ms (Post-Release)")

for N in N_vals:
    # Derive Space-Time Coupled Parameters
    h = L / N
    dt_coupled = h / c  # Enforces Courant Number = 1.0

    print(f"Running N={N:4d}, dt={dt_coupled:.3e} s")

    # Setup Params object
    p_coupled = BarParams(
        N=int(N),
        t_end=t_end,
        u0=u0,
        v0=v0,
        e=e,
        dt_factor=0.999,
        rho=rho,
        L=L,
        E=E,
        S=S,
    )

    # Run Simulations
    for name, sim in methods.items():
        r = sim(p_coupled)

        # Analytic Solution
        ana = analytic_bar_solution(r["t"], p_coupled)

        # Filter for Post-Release Only
        mask_post = r["t"] > 1.1 * t_release

        if np.any(mask_post):
            # ─── Position Error ───
            u_num = r["u"][mask_post]
            u_ana = ana["u"][mask_post]

            abs_diff_u = np.abs(u_num - u_ana)
            scale_u = np.mean(np.abs(u_ana))
            if scale_u < 1e-12:
                scale_u = 1.0

            err_u = np.mean(abs_diff_u) / scale_u

            # ─── Velocity Error ───
            v_num = r["v"][mask_post]
            v_ana = ana["v"][mask_post]

            abs_diff_v = np.abs(v_num - v_ana)
            scale_v = np.mean(np.abs(v_ana))
            if scale_v < 1e-12:
                scale_v = 1.0

            err_v = np.mean(abs_diff_v) / scale_v

            # Store
            results[name]["dt"].append(dt_coupled)
            results[name]["err_u"].append(err_u)
            results[name]["err_v"].append(err_v)

In [ ]:
# ─── Plotting ───

write = False

# Position Convergence
fig_u = go.Figure()
colors = {"Moreau-Jean": "lightblue", "CD-Lagrange": "#2072b2", "NS-Newmark": "red"}

for name, data in results.items():
    if not data["dt"]:
        continue
    fig_u.add_trace(
        go.Scatter(
            x=data["dt"],
            y=data["err_u"],
            mode="lines",
            name=name,
            line=dict(color=colors.get(name)),
        )
    )

# Reference Line (Position)
if results["Moreau-Jean"]["dt"]:
    ref_x = np.array(results["Moreau-Jean"]["dt"])
    mid = len(ref_x) // 2
    ref_y = ref_x * (results["Moreau-Jean"]["err_u"][mid] / ref_x[mid])
    fig_u.add_trace(
        go.Scatter(
            x=ref_x[10:20],
            y=ref_y[10:20],
            mode="lines",
            name="Order 1 Ref",
            line=dict(color="black", dash="dash", width=1),
        )
    )

    # Add O(dt^1/2) line
    ref_y_sq = np.sqrt(ref_x) * (
        results["Moreau-Jean"]["err_u"][mid] / np.sqrt(ref_x[mid])
    )
    fig_u.add_trace(
        go.Scatter(
            x=ref_x[10:20],
            y=ref_y_sq[10:20],
            mode="lines",
            name="Order 0.5 Ref",
            line=dict(color="gray", dash="dot", width=1),
        )
    )

fig_u.update_layout(get_layout())
fig_u.update_layout(
    title="Position Convergence (h = cΔt, Post-Release)",
    xaxis_title=r"$\Delta t \mathrm{ (s)}$",
    yaxis_title=r"$\eta_u$",
    xaxis=dict(type="log"),
    yaxis=dict(type="log", side="right"),
    template="plotly_white",
    height=500,
    width=700,
)
fig_u.show()
if write:
    fig_u.write_image(
        f"../output/impacting_bar_convergence_position_e{e}_dtf0.999.pdf",
        width=400,
        height=300,
    )


# Velocity Convergence
fig_v = go.Figure()

for name, data in results.items():
    if not data["dt"]:
        continue
    fig_v.add_trace(
        go.Scatter(
            x=data["dt"],
            y=data["err_v"],
            mode="lines",
            name=name,
            line=dict(color=colors.get(name)),
            showlegend=True,
        )
    )

# Reference Line (Velocity)
# We anchor the O(dt) line to the velocity error of Moreau-Jean
if results["Moreau-Jean"]["dt"]:
    ref_x = np.array(results["Moreau-Jean"]["dt"])
    mid = len(ref_x) // 2
    # Recalculate intercept for velocity data
    ref_y_v = ref_x * (results["Moreau-Jean"]["err_v"][mid] / ref_x[mid])
    fig_v.add_trace(
        go.Scatter(
            x=ref_x[10:20],
            y=ref_y_v[10:20],
            mode="lines",
            name="Order 1 Ref",
            line=dict(color="black", dash="dash", width=1),
        )
    )

    # Add O(dt^0.5) line
    ref_y_sqrt = np.sqrt(ref_x) * (
        results["Moreau-Jean"]["err_v"][mid] / np.sqrt(ref_x[mid])
    )
    fig_v.add_trace(
        go.Scatter(
            x=ref_x[10:20],
            y=ref_y_sqrt[10:20],
            mode="lines",
            name="Order 0.5 Ref",
            line=dict(color="gray", dash="dot", width=1),
        )
    )

fig_v.update_layout(get_layout())
fig_v.update_layout(
    title="Velocity Convergence (h = cΔt, Post-Release)",
    xaxis_title=r"$\Delta t \mathrm{ (s)}$",
    yaxis_title=r"$\eta_v$",
    xaxis=dict(type="log"),
    yaxis=dict(type="log", side="right"),
    template="plotly_white",
    height=500,
    width=700,
)
fig_v.show()

if write:
    fig_v.write_image(
        f"../output/impacting_bar_convergence_velocity_e{e}_dtf0.999.pdf",
        width=400,
        height=300,
    )